# SpaceX Falcon 9 First Stage Landing Prediction

SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each. Much of the savings is because SpaceX can reuse the first stage. Therefore, if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

In this lab, you will create a machine learning pipeline to predict if the first stage will land.

## Objectives

Perform exploratory Data Analysis and determine Training Labels:
- Create a column for the class
- Standardize the data
- Split into training data and test data

Find best Hyperparameter for SVM, Classification Trees and Logistic Regression:
- Find the method that performs best using test data

## Import Libraries and Define Auxiliary Functions

In [ ]:
# Install required packages if needed
# !pip install numpy pandas seaborn scikit-learn matplotlib

In [ ]:
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices
import numpy as np
# Matplotlib is a plotting library for Python
import matplotlib.pyplot as plt
# Seaborn is a Python data visualization library based on matplotlib
import seaborn as sns
# Preprocessing allows us to standardize our data
from sklearn import preprocessing
# Allows us to split our data into training and testing data
from sklearn.model_selection import train_test_split
# Allows us to test parameters of classification algorithms and find the best one
from sklearn.model_selection import GridSearchCV
# Logistic Regression classification algorithm
from sklearn.linear_model import LogisticRegression
# Support Vector Machine classification algorithm
from sklearn.svm import SVC
# Decision Tree classification algorithm
from sklearn.tree import DecisionTreeClassifier
# K Nearest Neighbors classification algorithm
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
def plot_confusion_matrix(y, y_predict):
    """This function plots the confusion matrix"""
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax)  # annot=True to annotate cells
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels(['did not land', 'land'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

## Load the Dataframe

In [ ]:
# Load dataset_part_2.csv
data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv')

In [ ]:
data.head()

In [ ]:
# Load dataset_part_3.csv (the feature-engineered dataset)
X = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv')

In [ ]:
X.head(100)

## TASK 1

Create a NumPy array from the column `Class` in `data`, by applying the method `to_numpy()` then assign it to the variable `Y`. Make sure the output is a Pandas series (only one bracket `df['name of column']`).

In [ ]:
# Extract the 'Class' column and convert it to a NumPy array
Y = data['Class'].to_numpy()

# Verify the type and contents
print("Type of Y:", type(Y))
print("First 10 elements of Y:", Y[:10])

## TASK 2

Standardize the data in `X` then reassign it to the variable `X` using the transform provided below.

In [ ]:
# Students get this
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)

We split the data into training and testing data using the function `train_test_split`. The training data is divided into validation data, a second set used for training data; then the models are trained and hyperparameters are selected using the function `GridSearchCV`.

## TASK 3

Use the function `train_test_split` to split the data X and Y into training and test data. Set the parameter `test_size` to 0.2 and `random_state` to 2. The training data and test data should be assigned to the following labels:

`X_train, X_test, Y_train, Y_test`

In [ ]:
from sklearn.model_selection import train_test_split

# TASK 3: Split the data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=2
)

# Verify the shape of the resulting datasets
print(f"X_train shape: {X_train.shape} (80% of data for training)")
print(f"X_test shape:  {X_test.shape} (20% of data for testing)")
print(f"Y_train shape: {Y_train.shape}")
print(f"Y_test shape:  {Y_test.shape}")

## TASK 4

Create a logistic regression object then create a `GridSearchCV` object `logreg_cv` with `cv = 10`. Fit the object to find the best parameters from the dictionary `parameters`.

In [ ]:
parameters = {'C': [0.01, 0.1, 1],
              'penalty': ['l2'],
              'solver': ['lbfgs']}

In [ ]:
parameters = {"C": [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}  # l1 lasso l2 ridge
lr = LogisticRegression()
logreg_cv = GridSearchCV(estimator=lr, cv=10, param_grid=parameters).fit(X_train, Y_train)

In [ ]:
print("tuned hpyerparameters :(best parameters) ", logreg_cv.best_params_)
print("accuracy :", logreg_cv.best_score_)

## TASK 5

Calculate the accuracy on the test data using the method `score`:

In [ ]:
# TASK 5: Calculate the accuracy on the test data
test_accuracy = logreg_cv.score(X_test, Y_test)

print("Accuracy on test data :", test_accuracy)

In [ ]:
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

Examining the confusion matrix, we see that logistic regression can distinguish between the different classes. We see that the problem is false positives.

Overview:
- **True Positive** - 12 (True label is landed, Predicted label is also landed)
- **False Positive** - 3 (True label is not landed, Predicted label is landed)

## TASK 6

Create a support vector machine object then create a `GridSearchCV` object `svm_cv` with `cv = 10`. Fit the object to find the best parameters from the dictionary `parameters`.

In [ ]:
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}

svm = SVC()
svm_cv = GridSearchCV(estimator=svm, cv=10, param_grid=parameters).fit(X_train, Y_train)

In [ ]:
print("tuned hpyerparameters :(best parameters) ", svm_cv.best_params_)
print("accuracy :", svm_cv.best_score_)

## TASK 7

Calculate the accuracy on the test data using the method `score`:

In [ ]:
# TASK 7: Calculate the accuracy of svm_cv on the test data
test_accuracy_svm = svm_cv.score(X_test, Y_test)

print("Accuracy on test data :", test_accuracy_svm)

In [ ]:
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 8

Create a decision tree classifier object then create a `GridSearchCV` object `tree_cv` with `cv = 10`. Fit the object to find the best parameters from the dictionary `parameters`.

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2, 4, 6, 8, 10],
              'max_features': ['auto', 'sqrt', 'log2'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}

tree = DecisionTreeClassifier()
tree_cv = GridSearchCV(estimator=tree, cv=10, param_grid=parameters).fit(X_train, Y_train)

In [ ]:
print("tuned hpyerparameters :(best parameters) ", tree_cv.best_params_)
print("accuracy :", tree_cv.best_score_)

## TASK 9

Calculate the accuracy of `tree_cv` on the test data using the method `score`:

In [ ]:
# TASK 9: Calculate the accuracy of tree_cv on the test data
test_accuracy_tree = tree_cv.score(X_test, Y_test)

print("Accuracy on test data :", test_accuracy_tree)

In [ ]:
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 10

Create a k nearest neighbors object then create a `GridSearchCV` object `knn_cv` with `cv = 10`. Fit the object to find the best parameters from the dictionary `parameters`.

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}

KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(estimator=KNN, cv=10, param_grid=parameters).fit(X_train, Y_train)

In [ ]:
print("tuned hpyerparameters :(best parameters) ", knn_cv.best_params_)
print("accuracy :", knn_cv.best_score_)

## TASK 11

Calculate the accuracy of `knn_cv` on the test data using the method `score`:

In [ ]:
# TASK 11: Calculate the accuracy of knn_cv on the test data
test_accuracy_knn = knn_cv.score(X_test, Y_test)

print("Accuracy on test data :", test_accuracy_knn)

In [ ]:
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 12

Find the method that performs best:

In [ ]:
# TASK 12: Compare all models and find the best one
model_scores = {
    'Logistic Regression': test_accuracy,
    'SVM':                 test_accuracy_svm,
    'Decision Tree':       test_accuracy_tree,
    'KNN':                 test_accuracy_knn
}

print("Test Accuracy of each model:")
print("-" * 40)
for model, score in model_scores.items():
    print(f"{model:<25}: {score:.4f}")

best_model = max(model_scores, key=model_scores.get)
print("-" * 40)
print(f"\nBest performing method: {best_model} with accuracy {model_scores[best_model]:.4f}")

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(8, 5))
models = list(model_scores.keys())
scores = list(model_scores.values())
colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']
bars = ax.bar(models, scores, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Test Accuracy Comparison of Classification Models')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Authors

IBM Skills Network